In [1]:
# Connect to postgis using geopandas
import psycopg2

import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

con = psycopg2.connect(database="track", user="postgres", host="localhost")

In [2]:
import geopandas as gpd

sql = "select * from species"
species = gpd.read_postgis(sql, con, geom_col='geom', index_col='gid')

In [3]:
# Select distinct species
print(f'Number of individuals: {len(species["id_individ"].unique())}')

Number of individuals: 14


In [4]:
# Group by individual and get max - min period
species['period'] = species.groupby('id_individ')['timestamp'].transform(lambda x: x.max() - x.min())

In [5]:
# Construct tracks
from shapely import LineString

tracks = species.groupby('id_individ').agg({'geom': lambda x: LineString(x.tolist())})
tracks = gpd.GeoDataFrame(tracks, geometry='geom')
tracks['animal_id'] = tracks.index
tracks.set_crs(epsg=32735, inplace=True)

,geom,animal_id
id_individ,,
1.0,"LINESTRING (487943.000 7923979.000, 452739.000...",1.0
2.0,"LINESTRING (500559.000 7885461.000, 513390.000...",2.0
3.0,"LINESTRING (492905.000 7908078.000, 489834.000...",3.0
4.0,"LINESTRING (476122.000 7932569.000, 485329.000...",4.0
5.0,"LINESTRING (492305.000 7913905.000, 491755.000...",5.0
6.0,"LINESTRING (496943.000 7905031.000, 497099.000...",6.0
7.0,"LINESTRING (466351.000 7943598.000, 464631.000...",7.0
8.0,"LINESTRING (464387.000 7899992.000, 463987.000...",8.0
9.0,"LINESTRING (477358.000 7904592.000, 490803.000...",9.0


In [6]:
# Plot tracks
tracks.explore(column='animal_id', cmap='Paired', tiles='CartoDB positron', attr='CartoDB', popup='animal_id')